# 전력수급 API 실습: 받아서 · 쌓고 · 계산하기

한국전력거래소가 공공데이터포털(OpenAPI)로 열어 둔 **5분 단위 전력수급**을 직접 받아 봅니다.
지난 세션이 "만들어진 대시보드가 어떻게 돌아가는가"였다면, 이번에는 **그 재료(Parquet)를 누가 매일 사다 놓는가**를 만듭니다.

> **오늘의 핵심 한 줄**  이 API는 '오늘'만 알려 줍니다. 어제 값은 다시 받을 수 없으니, **매일 받아서 쌓아 두는 일**을 자동화해야 합니다.

| 파트 | 내용 | 시간 |
|---|---|---|
| 1 | 인증키로 첫 호출, 응답 읽기 | 15분 |
| 2 | 오늘 하루치를 표로 만들고 검증·차트 | 20분 |
| 3 | 파일(Parquet)로 쌓기, 중복 없이 | 10분 |
| 4 | SQL로 계산하기 (pandas와 나란히) | 15분 |
| 5 | 대시보드에 연결 | 5분 |
| 6 | GitHub Actions로 자동화 | 10분 |
| 7 | 정리·체크리스트 (+ 심화 과제) | 5분 |

**오늘 쓰는 API** (모두 공공데이터포털 · 한국전력거래소)

| API | 돌려주는 것 | 응답 형식 | 쓰는 곳 |
|---|---|---|---|
| 현재전력수급현황조회_GW | 가장 최근 5분 한 건 | JSON · XML | Part 1 |
| 오늘전력수급현황조회_GW | 오늘 0시부터 지금까지 5분 단위 | XML | Part 2~6 (메인) |
| 전력거래실적 | 연료원별·시간별 전력거래량 (약 2개월 전까지) | JSON · XML | Part 7 심화 |

## 준비: 인증키 받기 (세션 전에 해 오세요)

1. [공공데이터포털](https://www.data.go.kr) 회원가입 · 로그인
2. 아래 두 API 페이지에서 각각 **활용신청** (개발계정은 자동 승인, 하루 100회까지 호출)
   * 「한국전력거래소_현재전력수급현황조회_GW」
   * 「한국전력거래소_오늘전력수급현황조회_GW」
3. 마이페이지 → **인증키 발급현황** → **일반 인증키(Decoding)** 복사

> 신청 직후에는 키가 아직 등록되지 않았다는 오류가 날 수 있습니다. 세션 전날까지 신청해 두세요.
> **키는 비밀번호입니다.** 노트북 셀에 직접 적지 말고 아래 입력창에 붙여넣으세요.
> `print(r.url)` 처럼 요청 주소를 찍으면 키가 함께 보이니 공유할 노트북에서는 쓰지 마세요.

In [ ]:
# 필요한 도구를 불러옵니다. (Colab에는 대부분 있고, duckdb만 설치합니다)
!pip -q install duckdb

import getpass, json, os
from io import StringIO
from urllib.parse import unquote

import duckdb
import pandas as pd
import plotly.express as px
import requests

# 인증키를 입력창에 붙여넣습니다. (화면에 보이지 않고, 노트북에도 저장되지 않습니다)
KEY = unquote(os.environ.get("KPX_API_KEY") or getpass.getpass("공공데이터포털 인증키(Decoding): ").strip())
print("키 길이:", len(KEY))

## Part 1. 첫 호출: 지금 이 순간의 전력수급 (15분)

API 호출은 **주소 + 요청변수**를 보내고 **응답**을 받는 일입니다. 브라우저 주소창에 넣는 것과 같은데, 파이썬이 대신 합니다.

| 요청변수 | 뜻 |
|---|---|
| `serviceKey` | 내 인증키 |
| `dataType` | 응답 형식 (`json` 또는 `xml`) |

In [ ]:
URL_NOW = "https://apis.data.go.kr/B552115/sukub5mMaxDatetime2/getSukub5mMaxDatetime2"

r = requests.get(URL_NOW, params={"serviceKey": KEY, "dataType": "json"}, timeout=30)
print(r.status_code)
print(r.text[:1000])

### 응답 읽는 법

응답은 크게 **header**(성공했나?)와 **body**(내용)로 나뉘고, 내용은 `body → items → item` 안에 있습니다.

| 항목 | 뜻 | 단위 |
|---|---|---|
| `baseDatetime` | 기준일시 | 년월일시분초 |
| `currPwrTot` | 현재수요 | MW |
| `suppAbility` | 공급능력 | MW |
| `forecastLoad` | 최대예측수요 | MW |
| `suppReservePwr` | 공급예비력 | MW |
| `suppReserveRate` | 공급예비율 | % |
| `operReservePwr` | 운영예비력 | MW |
| `operReserveRate` | 운영예비율 | % |
| `rn` | 순번 | |

In [ ]:
js = r.json()
root = js.get("response", js)          # 'response' 껍데기가 있을 수도, 없을 수도 있어요
print("header:", root["header"])

item = root["body"]["items"]["item"]   # 한 건이면 dict, 여러 건이면 list
pd.json_normalize(item)                # 어느 쪽이든 표로 바뀝니다

> 경로가 맞지 않아 `KeyError` 가 나면 `print(js)` 로 전체를 찍어서 눈으로 길을 따라가 보세요. 응답의 모양을 읽는 것이 API 실습의 절반입니다.

### 연습 1-A. 일부러 틀려 보기

키를 틀리게 보내면 어떤 응답이 올까요? 직접 불러서 읽어 보세요. (하루 100회 한도에 포함되니 한 번만!)

In [ ]:
bad = requests.get(URL_NOW, params={"serviceKey": "wrong-key", "dataType": "json"}, timeout=30)
print(bad.status_code)
print(bad.text[:500])

<details><summary>정리</summary>

* **오류도 '응답'으로 옵니다.** 게이트웨이 오류는 `status_code` 가 200이 아닐 수도 있고, 본문에 오류 코드가 담겨 올 수도 있습니다.
  (인증키 오류는 보통 `SERVICE_KEY_IS_NOT_REGISTERED_ERROR` 같은 문구로 옵니다.)
* 그래서 자동화 스크립트는 **"받은 내용이 정말 기대한 모양인가"**를 반드시 확인합니다. (Part 2 검증, Part 6에서 다시 만납니다)

</details>

## Part 2. 오늘 하루치를 표로 (20분)

「오늘전력수급현황조회」는 **오늘 0시부터 지금까지 5분 간격**으로 돌려줍니다. 하루는 5분 × 288칸이라 `numOfRows=400` 이면 한 번에 다 받습니다.
이 API는 응답 형식이 **XML**입니다. (JSON과 모양만 다르고 내용은 같습니다.)

In [ ]:
URL_TODAY = "https://apis.data.go.kr/B552115/Sukub5mToday/getSukub5mToday"

r = requests.get(URL_TODAY, params={"serviceKey": KEY, "dataType": "xml", "pageNo": 1, "numOfRows": 400}, timeout=30)
print(r.status_code)
print(r.text[:900])

**키가 아직 안 되는 분만** 아래 셀을 실행하세요. 명세의 항목 이름을 따라 만든 **연습용 가짜 응답**입니다(실제 값이 아닙니다). 이어지는 셀이 그대로 돕니다.
(Part 1은 강사 화면으로 함께 보고, 키가 되면 다시 해 보세요. Part 7의 심화 셀은 실제 키가 있어야 합니다.)

In [ ]:
# ⚠️ 키가 아직 승인되지 않은 분만 실행하세요. 연습용 가짜 응답입니다.
import math
from datetime import datetime, timedelta

def _fake_item(i, day0):
    t = day0 + timedelta(minutes=5 * i)
    demand = 62000 + 14000 * math.sin((i / 12 - 6) / 24 * 2 * math.pi)
    supply = 92000
    f = {"baseDatetime": t.strftime("%Y%m%d%H%M%S"), "currPwrTot": f"{demand:.1f}", "suppAbility": f"{supply:.1f}",
         "forecastLoad": "77000.0", "suppReservePwr": f"{supply - demand:.1f}",
         "suppReserveRate": f"{(supply - demand) / demand * 100:.2f}",
         "operReservePwr": f"{supply - demand - 5000:.1f}", "operReserveRate": f"{(supply - demand - 5000) / demand * 100:.2f}"}
    return "<item>" + "".join(f"<{k}>{v}</{k}>" for k, v in f.items()) + "</item>"

_day0 = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
_n = int((datetime.now() - _day0).total_seconds() // 300)
class r:       # requests 응답처럼 .text 만 있으면 아래 셀이 그대로 동작합니다
    text = "<response><header><resultCode>00</resultCode></header><body><items>" + "".join(_fake_item(i, _day0) for i in range(_n)) + "</items></body></response>"

### XML → 표(DataFrame)

`pandas.read_xml` 이 `<item>` 하나하나를 한 행으로 바꿔 줍니다.

In [ ]:
raw = pd.read_xml(StringIO(r.text), xpath=".//item", parser="etree", dtype=str)
print(raw.shape)
raw.head()

### 이름표 붙이고 자료형 바꾸기

* 영문 항목 이름을 알아보기 쉬운 열 이름으로 바꿉니다. (MW, % 는 이름에 남겨 둡니다)
* 기준일시는 '년월일시분초'라서 **숫자만 남긴 뒤** 날짜·시각으로 바꿉니다. 형식이 `20261031143000` 이든 `2026-10-31 14:30:00` 이든 읽힙니다.
* 나머지는 숫자로 바꿉니다. 바뀌지 않는 값은 `NaN` 이 됩니다.

In [ ]:
NAMES = {
    "baseDatetime": "ts",                 # 기준일시
    "currPwrTot": "demand_mw",            # 현재수요
    "suppAbility": "supply_mw",           # 공급능력
    "forecastLoad": "forecast_mw",        # 최대예측수요
    "suppReservePwr": "reserve_mw",       # 공급예비력
    "suppReserveRate": "reserve_pct",     # 공급예비율(%)
    "operReservePwr": "oper_reserve_mw",  # 운영예비력
    "operReserveRate": "oper_reserve_pct" # 운영예비율(%)
}

df = raw[list(NAMES)].rename(columns=NAMES)

digits = df["ts"].astype(str).str.replace(r"\D", "", regex=True).str.ljust(14, "0").str[:14]
df["ts"] = pd.to_datetime(digits, format="%Y%m%d%H%M%S")
for c in df.columns.drop("ts"):
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.sort_values("ts").reset_index(drop=True)
df.info()

### 믿기 전에 확인하기 (검증)

받은 값을 그대로 믿고 쌓으면, 한 번 틀린 값이 몇 달 뒤 그래프에서 발견됩니다. **쌓기 전에 점검하는 함수**를 만들어 둡니다.

In [ ]:
def check(df):
    gaps = df["ts"].diff().dropna()
    print("행 수            :", len(df))
    print("시각 범위        :", df["ts"].min(), "~", df["ts"].max())
    print("중복된 시각      :", int(df["ts"].duplicated().sum()))
    print("5분이 아닌 간격  :", int((gaps != pd.Timedelta(minutes=5)).sum()))
    print("수요가 비었거나 0:", int((df["demand_mw"].isna() | (df["demand_mw"] <= 0)).sum()))
    print("수요 범위(MW)    :", df["demand_mw"].min(), "~", df["demand_mw"].max())

check(df)

### 연습 2-A. 결과 읽기

* 가장 마지막 시각은 몇 시인가요? 지금 시각과 비슷한가요?
* '수요가 비었거나 0' 이 0보다 크다면 어떤 행일까요? (하루 288칸이 미리 만들어져 있고, 아직 오지 않은 시각은 비어 있을 수도 있습니다)
* 5분이 아닌 간격이 있다면 어디일까요?

### 연습 2-B. 실적이 있는 행만 남기기

`demand_mw` 가 비어 있거나 0 이하인 행을 걸러서 `df` 를 다시 만들어 보세요.

In [ ]:
# 여기에 직접 작성해 보세요.

<details><summary>정답 보기</summary>

```python
df = df[df["demand_mw"].notna() & (df["demand_mw"] > 0)].reset_index(drop=True)
check(df)
```

</details>

In [ ]:
# 연습 2-B를 건너뛴 분도 이어서 할 수 있게 같은 처리를 한 번 더 해 둡니다. (이미 걸러졌다면 아무 일도 일어나지 않습니다)
df = df[df["demand_mw"].notna() & (df["demand_mw"] > 0)].reset_index(drop=True)
fig = px.line(df, x="ts", y=["supply_mw", "demand_mw"], labels={"value": "MW", "ts": "시각", "variable": ""})
fig.show()

> **AI와 함께 일할 때** API 코드를 Claude에게 맡기려면, 말로 설명하기보다 **API 명세(요청변수·응답항목)와 응답 한 건**을 그대로 붙여 주세요. 항목 이름과 단위가 정확히 전달돼서 코드가 훨씬 정확해집니다. 그리고 어떤 도구가 쓴 코드든 위의 `check()` 같은 **검증은 사람이 정합니다.**

## Part 3. 파일로 쌓기 (10분)

받은 표를 **Parquet** 파일로 저장합니다. (지난 세션: Parquet = 계산 재료, 작고 빠른 표 파일)

In [ ]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)
PATH = Path("data/supply_5min.parquet")
df.to_parquet(PATH, index=False)

print(PATH.stat().st_size, "bytes")
pd.read_parquet(PATH).shape

### 같은 걸 또 쌓으면?

자동화는 **같은 일을 두 번 돌려도 결과가 같아야** 합니다. 그냥 이어 붙이면 어떻게 되는지 봅니다.

In [ ]:
bad = pd.concat([pd.read_parquet(PATH), df])
print("행 수:", len(bad), "/ 중복된 시각:", int(bad["ts"].duplicated().sum()))

**같은 시각(`ts`)은 새로 받은 값으로 덮어쓰기**로 해결합니다. (값이 나중에 정정될 수도 있으니 새 값이 우선입니다)

In [ ]:
def upsert(old, new):
    """같은 시각(ts)은 새 값으로 덮어쓰고, 시각 순으로 정렬합니다."""
    both = new if old is None else pd.concat([old, new])
    return both.drop_duplicates("ts", keep="last").sort_values("ts").reset_index(drop=True)

merged = upsert(pd.read_parquet(PATH), df)
print("행 수:", len(merged), "/ 중복된 시각:", int(merged["ts"].duplicated().sum()))

### 연습 3-A. 내일이 되면?

API는 내일이 되면 어제 값을 주지 않습니다. 그래서 **어제 파일에 오늘 것을 얹는** 방식이 필요합니다.
아래 `yesterday` 는 "어제 받아 둔 파일"이라고 가정한 표입니다. `upsert` 로 `df`(오늘)와 합쳐 `store` 를 만들고, 몇 일치가 들어 있는지 세어 보세요.

In [ ]:
yesterday = df.assign(ts=df["ts"] - pd.Timedelta(days=1))   # 어제 받아 둔 파일이라고 가정

# 여기에 직접 작성해 보세요.

<details><summary>정답 보기</summary>

```python
store = upsert(yesterday, df)
print(store["ts"].dt.date.nunique(), "일,", len(store), "행")
```

</details>

In [ ]:
# 연습 3-A를 건너뛴 분도 이어서 할 수 있게 같은 걸 만들어 둡니다.
yesterday = df.assign(ts=df["ts"] - pd.Timedelta(days=1))
store = upsert(yesterday, df)
print(store["ts"].dt.date.nunique(), "일,", len(store), "행")

## Part 4. SQL로 계산하기 (15분)

지난 세션에서 본 대로 **SQL은 DB에게 보내는 주문서이고, pandas와 같은 일을 합니다.** DuckDB는 파이썬 표(DataFrame)를 이름만 부르면 바로 읽습니다.

며칠치가 쌓여야 재미있는 계산이 됩니다. 강사가 며칠 쌓아 둔 파일 주소를 알려 주면 `DATA_URL` 에 넣으세요. 비워 두면 Part 3에서 만든 `store` 를 씁니다.

In [ ]:
DATA_URL = ""   # 강사가 알려 주는 주소 (예: https://…/data/supply_5min.parquet)

supply = pd.read_parquet(DATA_URL) if DATA_URL else store
print(supply["ts"].dt.date.nunique(), "일,", len(supply), "행")

### pandas와 SQL, 같은 질문 두 가지 답

**질문: 하루 중 가장 전기를 많이 쓴 순간 5개는?**

In [ ]:
# pandas
supply.nlargest(5, "demand_mw")[["ts", "demand_mw", "supply_mw", "reserve_pct"]]

In [ ]:
# SQL (표 이름은 변수 이름 supply 를 그대로 씁니다)
duckdb.sql("""
SELECT ts, demand_mw, supply_mw, reserve_pct
FROM supply
ORDER BY demand_mw DESC
LIMIT 5
""").df()

**질문: 날짜별 최대수요는?**  (`GROUP BY` = pandas의 `groupby`)

In [ ]:
supply.groupby(supply["ts"].dt.date)["demand_mw"].max()

In [ ]:
duckdb.sql("""
SELECT CAST(ts AS DATE) AS 날짜, max(demand_mw) AS 최대수요_MW
FROM supply
GROUP BY 1
ORDER BY 1
""").df()

### 연습 4-A. 일별 최대수요와 **그 시각**

날짜별 최대수요가 **몇 시 몇 분**에 나왔는지도 같이 보여 주세요. (힌트: `arg_max(ts, demand_mw)` = 수요가 가장 클 때의 `ts`)

In [ ]:
# 여기에 직접 작성해 보세요.

<details><summary>정답 보기</summary>

```python
duckdb.sql("""
SELECT CAST(ts AS DATE) AS 날짜,
       max(demand_mw) AS 최대수요_MW,
       strftime(arg_max(ts, demand_mw), '%H:%M') AS 최대수요_시각
FROM supply
GROUP BY 1
ORDER BY 1
""").df()
```

</details>

### 연습 4-B. 시간대별 평균 수요 그래프

`hour(ts)` 로 시(0~23)를 뽑아 시간대별 평균 수요를 구하고, `px.line` 으로 그려 보세요. 하루 중 언제 전기를 많이 쓰나요?

In [ ]:
# 여기에 직접 작성해 보세요.

<details><summary>정답 보기</summary>

```python
hourly = duckdb.sql("""
SELECT hour(ts) AS 시, avg(demand_mw) AS 평균수요_MW
FROM supply
GROUP BY 1
ORDER BY 1
""").df()
px.line(hourly, x="시", y="평균수요_MW", markers=True)
```

</details>

### 연습 4-C (선택). 빠진 곳 찾기

자동 수집은 가끔 놓칩니다. **5분보다 간격이 큰 곳**을 찾아 보세요. (힌트: `lag(ts) OVER (ORDER BY ts)` = 바로 앞 행의 `ts`, `date_diff('minute', 앞, 뒤)`)

In [ ]:
# 여기에 직접 작성해 보세요.

<details><summary>정답 보기</summary>

```python
duckdb.sql("""
SELECT prev_ts AS 직전_시각, ts AS 다음_시각, date_diff('minute', prev_ts, ts) AS 간격_분
FROM (SELECT ts, lag(ts) OVER (ORDER BY ts) AS prev_ts FROM supply)
WHERE date_diff('minute', prev_ts, ts) > 5
ORDER BY ts
""").df()
```
하루가 끝나기 전에 받는 구조라 **밤 11시쯤부터 자정까지**는 비는 것이 정상입니다. (Part 6 참고)

</details>

## Part 5. 대시보드에 연결 (5분)

강사가 배포한 대시보드를 열어 보세요. 방금 만든 것과 **같은 모양의 Parquet**(`supply_5min.parquet`)을 브라우저가 내려받아, 브라우저 안의 DuckDB가 계산해서 그래프를 그립니다.

```
API ──(Actions가 매일 호출)──▶ site/data/supply_5min.parquet ──▶ 브라우저(DuckDB-Wasm)가 읽고 계산 ──▶ 그래프
```

대시보드 아래쪽 **'직접 SQL 써보기'** 에 Part 4에서 쓴 SQL을 그대로 붙여 넣어 보세요. 표 이름이 같은 `supply` 라서 **파이썬에서도, 브라우저에서도 같은 SQL이 돕니다.**

In [ ]:
# (선택) 내 컴퓨터에서 대시보드를 돌려 보고 싶다면 파일을 내려받아 site/data/ 에 넣으세요.
from google.colab import files
files.download("data/supply_5min.parquet")

## Part 6. GitHub Actions로 자동화 (10분, 강사 시연 위주)

지금까지 손으로 한 일(호출 → 표 → 검증 → 쌓기)을 **저장소의 스크립트가 매일 대신** 합니다.

| 노트북에서 한 일 | 스크립트에서 | 파일 |
|---|---|---|
| `requests.get(...)` 호출 | `kpx.get_all(...)` (여러 페이지·오류·재시도 처리) | `scripts/kpx.py` |
| 이름표·자료형 | `to_frame(...)` | `scripts/fetch_today.py` |
| `check(df)` 검증 | `validate(...)` — 이상하면 **멈춤(빨간 X)** | `scripts/fetch_today.py` |
| `upsert(old, new)` | `drop_duplicates("ts", keep="last")` | `scripts/fetch_today.py` |

### 순서 (시연)

1. 저장소 **Settings → Secrets and variables → Actions → New repository secret** 에 이름 `KPX_API_KEY`, 값은 인증키로 등록
2. **Settings → Actions → General → Workflow permissions** 를 *Read and write* 로
3. **Settings → Pages → Source** 를 *GitHub Actions* 로
4. **Actions** 탭 → `collect-supply` → **Run workflow**
5. 로그에서 `받은 행: …`, `저장: …` 확인 → 저장소에 `data: 전력수급 5분 데이터 갱신` 커밋이 생기고, 이어서 `deploy` 작업이 대시보드(`https://<계정>.github.io/<저장소>/`)를 갱신합니다

### 워크플로 읽어 보기 (`.github/workflows/daily.yml`)

```yaml
on:
  schedule:
    - cron: "17 13 * * *"   # 13:17 UTC = 22:17 KST
    - cron: "37 14 * * *"   # 14:37 UTC = 23:37 KST
  workflow_dispatch:         # 손으로 Run workflow

...
      - name: 오늘 전력수급 받아서 쌓기
        env:
          KPX_API_KEY: ${{ secrets.KPX_API_KEY }}   # 코드에 키를 적지 않고 금고(secret)에서 꺼내 씀
        run: python scripts/fetch_today.py
```

* **왜 하루 두 번?** '오늘'만 주는 API라서 하루가 끝나기 전에 받아야 합니다. 예약 실행은 몇십 분 늦어질 수 있어, 한 번만 돌리면 자정을 넘겨 **그날을 통째로 놓칠** 수 있습니다. 두 번 돌려도 같은 시각은 덮어쓰므로 중복이 없습니다.
* 그래도 **밤 11시 반쯤부터 자정까지**는 비게 됩니다. (연습 4-C에서 본 간격)
* **바뀐 게 없으면 커밋하지 않습니다.**
* 수집 → 커밋 → **배포(GitHub Pages)** 가 한 워크플로 안에서 이어집니다. 수집이 실패하면 배포는 건너뜁니다.
* 검증에 실패하면 Actions가 **빨간 X** 로 알려 줍니다. *조용히 틀리는 자동화*가 가장 위험합니다.

## Part 7. 정리 (5분)

### 자동화 체크리스트

- [ ] 인증키를 코드·노트북에 쓰지 않았나? (secret, 환경변수)
- [ ] 같은 일을 두 번 돌려도 결과가 같은가? (덮어쓰기, 중복 제거)
- [ ] 이상한 값이 오면 **쌓기 전에 멈추는가**? (검증)
- [ ] 바뀐 게 없으면 아무것도 하지 않는가?
- [ ] 놓친 곳을 **나중에 찾아낼 수 있는가**? (연습 4-C)
- [ ] 하루 호출 한도(개발계정 100회)를 넘지 않는가?
- [ ] 파일이 계속 커진다는 것을 알고 있는가? (몇 달 단위로 파일을 나누는 방법도 있습니다)

### 이 API의 한계

* 과거 값은 이 API로 받을 수 없습니다. (과거 자료는 전력거래소 사이트의 파일 내려받기를 쓰거나, 오늘부터 쌓아 갑니다)
* 이 값은 5분 단위 **실시간 현황**입니다. 일별 확정 통계(예: EPSIS 전력수급실적)와 숫자가 정확히 같다고 가정하지 말고, 비교해서 확인하세요.
* 이 파일은 우리가 쌓기 시작한 날부터만 존재합니다. **데이터는 수집을 시작한 날이 곧 시작일**입니다.

### 심화 (선택). 연료원별 전력거래실적 API

같은 방식으로 다른 API도 받을 수 있습니다. 「한국전력거래소_전력거래실적」은 **하루 · 연료원 · 시간별 설비용량과 전력거래량**을 주며, 정산 때문에 **약 2개월 전 실적까지만** 나옵니다. (이 API는 이 노트북의 키와 별도로 **활용신청**이 필요합니다)

`tradeDay` 에 날짜를 넣어 한 날만 받아 봅니다. 응답이 비면 날짜를 한 달씩 앞으로 당겨 보세요.

In [ ]:
URL_TRADING = "https://apis.data.go.kr/B552115/PowerTradingResultInfo1/getPowerTradingResultInfo1"

day = (pd.Timestamp.today().to_period("M") - 2).to_timestamp() + pd.Timedelta(days=14)   # 2개월 전 15일
r = requests.get(URL_TRADING, params={"serviceKey": KEY, "dataType": "json", "pageNo": 1, "numOfRows": 500,
                                      "tradeDay": day.strftime("%Y%m%d")}, timeout=30)
js = r.json()
root = js.get("response", js)
print(root["header"], "/ 전체 건수:", root["body"].get("totalCount"))

trade = pd.json_normalize(root["body"]["items"]["item"])
trade["mgo"] = pd.to_numeric(trade["mgo"], errors="coerce")      # 전력거래량(MWh)
trade.groupby("fuel")["mgo"].sum().sort_values(ascending=False)

> 전체 건수가 `numOfRows` 보다 크면 `pageNo` 를 올려 가며 받아야 합니다. `scripts/kpx.py` 의 `get_all` 이 그 일을 합니다.

---

### 이번에 다루지 않은 것

화면에 보이지만 API가 없는 자료(예: 일별 전력수급실적)는 화면 뒤의 주소를 찾아 받아 오는 방법이 있습니다. 하지만 **비공식이라 언제든 깨질 수 있고** 이번 세션에는 내용이 많아 보류했습니다.
저장소의 `extras/scraping/` 에 지난 준비물을 그대로 남겨 두었습니다.

수고하셨습니다. 🙌